In [1]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sqlite3 # library for operating SQLite database

# 连接sqlite数据库
db_path = "/kaggle/input/datasets/tclaim/retail-total-2025-6-13/retail_total.db"
conn = sqlite3.connect(db_path)

# 定义查询函数
def query_db(sql):
    """
    执行SQL查询，返回pandas DataFrame
    :参数sql: 待执行的SQL语句
    :return: 查询结果DataFrame
    """
    # 执行查询
    return pd.read_sql(sql, conn)

print("ready")

ready


In [2]:
# 数据库各表表名
sql = "SELECT name FROM sqlite_master WHERE type='table';"
df = query_db(sql)
names = (df['name'].tolist())
print("names:",names)

names: ['sales_order_2025_6_13', 'customer', 'goods', 'shop']


In [3]:
# 每个表的各个列名
for name in names:
    sql = f'''SELECT * FROM {name} LIMIT 0;'''
    print(name,":",query_db(sql).columns.tolist())

sales_order_2025_6_13 : ['订单ID', '门店ID', '商品ID', '客户ID', '销售数量', '单价', '销售日期', '支付方式', '省份']
customer : ['客户ID', '客户姓名', '手机号', '注册日期', '会员等级', '所在城市']
goods : ['商品ID', '商品名称', '商品品牌', '商品类别', '建议零售价', '成本价']
shop : ['门店ID', '门店名称', '所在省份', '所在城市', '开店日期', '门店类型', '负责人']


## 案例 1：窗口函数 ROW_NUMBER 分组排名
**业务需求**：每个省份内，按订单总金额排名\
**思路解析**：`PARTITION BY`按省份分区，`ORDER BY`组内排序，生成行号排名

In [4]:
sql = '''
SELECT 
  订单ID,
  省份,
  销售数量 * 单价 AS 订单总金额,
  ROW_NUMBER() OVER(PARTITION BY 省份 ORDER BY 销售数量 * 单价 DESC) AS 省内排名
FROM sales_order_2025_6_13 
ORDER BY 省份, 省内排名;
'''
query_db(sql)

,订单ID,省份,订单总金额,省内排名
0,OD128,上海,40691.22,1
1,OD272,上海,34970.64,2
2,OD493,上海,33706.89,3
3,OD494,上海,20280.66,4
4,OD190,上海,17739.96,5
...,...,...,...,...
495,OD008,黑龙江,4041.66,11
496,OD076,黑龙江,3936.48,12
497,OD420,黑龙江,3477.45,13
498,OD086,黑龙江,1755.13,14


**结果说明**：每个省份单独从 1 开始排名，保留所有明细行

## 案例 2：窗口函数 累计求和 SUM () OVER ()
**业务需求**：按省份、日期统计累计销量\
**思路解析**：分区 + 有序累计，查看销量增长趋势

In [5]:
sql = '''
SELECT 
  订单ID,
  省份,
  销售日期,
  销售数量,
  SUM(销售数量) OVER(PARTITION BY 省份 ORDER BY 销售日期) AS 累计销量
FROM sales_order_2025_6_13
ORDER BY 省份, 销售日期;
'''
query_db(sql)

,订单ID,省份,销售日期,销售数量,累计销量
0,OD128,上海,2025-02-01 00:00:00,93,93
1,OD341,上海,2025-02-06 00:00:00,33,126
2,OD494,上海,2025-02-27 00:00:00,34,160
3,OD358,上海,2025-04-06 00:00:00,15,175
4,OD185,上海,2025-04-24 00:00:00,27,202
...,...,...,...,...,...
495,OD384,黑龙江,2025-04-23 00:00:00,2,457
496,OD178,黑龙江,2025-05-05 00:00:00,31,494
497,OD403,黑龙江,2025-05-05 00:00:00,6,494
498,OD154,黑龙江,2025-05-26 00:00:00,94,588


**结果说明**：逐行展示各省份当日及之前的合计销量

## 案例 3：并列排名 RANK / DENSE_RANK
**业务需求**：对商品类别平均单价做两种排名\
**思路解析**：`RANK`跳号排名，`DENSE_RANK`连续排名

In [6]:
sql = '''
WITH goods_avg AS (
  SELECT 商品类别, ROUND(AVG(建议零售价),2) AS 平均单价
  FROM goods GROUP BY 商品类别
)
SELECT 
  商品类别,
  平均单价,
  RANK() OVER(ORDER BY 平均单价 DESC) AS 跳号排名,
  DENSE_RANK() OVER(ORDER BY 平均单价 DESC) AS 连续排名
FROM goods_avg;
'''
query_db(sql)

,商品类别,平均单价,跳号排名,连续排名
0,数码家电,630.39,1,1
1,服饰鞋包,600.71,2,2
2,家居家纺,591.17,3,3
3,运动户外,578.03,4,4
4,美妆个护,291.95,5,5
5,母婴用品,161.12,6,6
6,生鲜果蔬,91.93,7,7
7,食品饮料,56.44,8,8
8,文具办公,43.13,9,9
9,日用百货,34.06,10,10


**结果说明**：相同单价名次一致，两种排名规则对比明显

## 案例 4：综合复杂查询（各省销量 TOP3 门店）
**业务需求**：查询每个省份销量前三名的门店信息\
**思路解析**：CTE + 多表关联 + 窗口排名 + 条件筛选，综合高阶用法

In [7]:
sql = '''
WITH shop_total AS (
  SELECT 
    o.门店ID, s.门店名称, s.所在省份, SUM(o.销售数量) AS 总销量
  FROM sales_order_2025_6_13 o
  LEFT JOIN shop s ON o.门店ID = s.门店ID
  GROUP BY o.门店ID, s.门店名称, s.所在省份
),
shop_rank AS (
  SELECT 
    *,
    ROW_NUMBER() OVER(PARTITION BY 所在省份 ORDER BY 总销量 DESC) AS 排名
  FROM shop_total
)
SELECT 所在省份,排名,门店ID,门店名称,总销量
FROM shop_rank
WHERE 排名 <= 3
ORDER BY 所在省份,排名;
'''
query_db(sql)

,所在省份,排名,门店ID,门店名称,总销量
0,上海,1,SH413,上海天河店,93
1,上海,2,SH363,上海青羊店,78
2,上海,3,SH275,上海姑苏店,63
3,云南,1,SH077,临沧姑苏店,188
4,云南,2,SH222,曲靖科创店,130
...,...,...,...,...,...
88,青海,2,SH207,玉树历下店,150
89,青海,3,SH049,西宁管城店,143
90,黑龙江,1,SH008,鹤岗天河店,105
91,黑龙江,2,SH378,佳木斯历下店,94


**结果说明**：输出各省份销量前三的核心门店

## 案例 5：SQL 性能优化（低效 VS 高效对比）
**业务需求**：查询 2025 年 3 月订单，对比两种写法性能差异\
**思路解析**：
1. 低效：`SELECT *` + 字段套函数
2. 高效：指定字段 + 范围查询

In [8]:
## 【低效写法】
sql = '''
SELECT * FROM sales_order_2025_6_13
WHERE strftime('%Y', 销售日期) = '2025' AND strftime('%m', 销售日期) = '03';
'''
print(query_db(sql))

## 【高效写法】生产规范
sql = '''
SELECT 订单ID,门店ID,销售数量,单价,销售日期
FROM sales_order_2025_6_13
WHERE 销售日期 >= '2025-03-01' AND 销售日期 < '2025-04-01';
'''
query_db(sql)

      订单ID   门店ID  商品ID  客户ID  销售数量      单价                 销售日期 支付方式  省份
0    OD004  SH242  G011  C368    26  543.87  2025-03-02 00:00:00  云闪付  新疆
1    OD011  SH036  G472  C101    54  576.47  2025-03-23 00:00:00  云闪付  甘肃
2    OD015  SH287  G129  C444    10  902.36  2025-03-27 00:00:00  云闪付  西藏
3    OD023  SH114  G070  C475    41  878.39  2025-03-03 00:00:00   微信  天津
4    OD024  SH026  G317  C382    26  733.78  2025-03-05 00:00:00  银行卡  湖南
..     ...    ...   ...   ...   ...     ...                  ...  ...  ..
104  OD489  SH101  G155  C296    81  607.06  2025-03-09 00:00:00  银行卡  河北
105  OD491  SH439  G011  C405    43  543.87  2025-03-27 00:00:00  云闪付  陕西
106  OD496  SH088  G212  C474    33  592.22  2025-03-30 00:00:00  银行卡  四川
107  OD498  SH102  G046  C037    90  137.00  2025-03-20 00:00:00  支付宝  湖南
108  OD499  SH307  G220  C379    16  634.56  2025-03-12 00:00:00   微信  安徽

[109 rows x 9 columns]


,订单ID,门店ID,销售数量,单价,销售日期
0,OD004,SH242,26,543.87,2025-03-02 00:00:00
1,OD011,SH036,54,576.47,2025-03-23 00:00:00
2,OD015,SH287,10,902.36,2025-03-27 00:00:00
3,OD023,SH114,41,878.39,2025-03-03 00:00:00
4,OD024,SH026,26,733.78,2025-03-05 00:00:00
...,...,...,...,...,...
104,OD489,SH101,81,607.06,2025-03-09 00:00:00
105,OD491,SH439,43,543.87,2025-03-27 00:00:00
106,OD496,SH088,33,592.22,2025-03-30 00:00:00
107,OD498,SH102,90,137.00,2025-03-20 00:00:00
